In [1]:
import pandas as pd 
import string 
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.layers import (
    Embedding,
    SimpleRNN,
    LSTM,
    GRU,
    Dense,
    Dropout
)
from tensorflow.keras.models import Sequential

2026-05-31 06:17:16.354849: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780208236.624190      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780208236.698356      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780208237.328642      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780208237.328685      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780208237.328689      58 computation_placer.cc:177] computation placer alr

In [2]:

df = pd.read_csv(
    '/kaggle/input/datasets/organizations/uciml/sms-spam-collection-dataset/spam.csv',
    encoding='latin-1')

In [3]:
df=df[['v1','v2']]
sms=df['v2']
cat=df['v1']

In [4]:
sms=sms.str.lower()
sms= sms.str.translate(str.maketrans('', '', string.punctuation))
sms.head()

0    go until jurong point crazy available only in ...
1                              ok lar joking wif u oni
2    free entry in 2 a wkly comp to win fa cup fina...
3          u dun say so early hor u c already then say
4    nah i dont think he goes to usf he lives aroun...
Name: v2, dtype: object

In [5]:
# vocab_size= len(tok.word_index)+1

# This is how you find your vocabulary size

In [6]:
vocab_size=9600 
tok= Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tok.fit_on_texts(sms)

In [7]:
list(tok.word_index.items())[:5] # make list from dict and print first 5 five items 

[('<OOV>', 1), ('to', 2), ('i', 3), ('you', 4), ('a', 5)]

In [8]:
tok.word_index["<OOV>"]

1

In [9]:
sequences=tok.texts_to_sequences(sms)
max_length= max(len(seq) for seq in sequences)

In [10]:
for i in range(3):
    print(sequences[i])
max_length

[47, 442, 4368, 797, 712, 679, 65, 10, 1250, 91, 121, 353, 1251, 154, 2897, 1252, 68, 58, 4369, 138]
[49, 312, 1403, 443, 7, 1837]
[51, 459, 10, 22, 5, 749, 903, 2, 181, 1838, 1136, 622, 1839, 2221, 263, 2222, 72, 1838, 2, 1840, 2, 313, 459, 2898, 80, 2899, 381, 2900]


171

In [11]:
padded_sequences= pad_sequences(sequences,  maxlen=max_length,
    padding="pre",)

In [12]:
print(padded_sequences[0])

[   0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0   47  442 4368
  797  712  679   65   10 1250   91  121  353 1251  154 2897 1252   68
   58 4369  138]


In [13]:
encoder= LabelEncoder()
cat_encoded=encoder.fit_transform(cat)
cat_encoded[:5] # print first five elements 

array([0, 0, 1, 0, 0])

In [14]:
X_train, X_test, y_train, y_test= train_test_split(
    padded_sequences, cat_encoded, test_size=0.2, random_state=42
)

In [15]:
y_test

array([0, 0, 1, ..., 0, 0, 1], shape=(1115,))

In [16]:
model = Sequential([
        Embedding(
            input_dim=vocab_size,
            output_dim=64,
            input_length=max_length
        ),

        LSTM(64),

        Dense(32, activation="relu"),

        Dropout(0.3),

        Dense(1, activation="sigmoid")
    ])



/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
2026-05-31 06:17:33.967653: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [17]:
    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

In [19]:
model.fit(
    X_train,
    y_train,
    epochs=3,
    batch_size=32,
    validation_split=0.1
)

loss, acc = model.evaluate(
    X_test,
    y_test,
    verbose=0
)


Epoch 1/3
126/126 ━━━━━━━━━━━━━━━━━━━━ 11s 67ms/step - accuracy: 0.9200 - loss: 0.2380 - val_accuracy: 0.9574 - val_loss: 0.1049
Epoch 2/3
126/126 ━━━━━━━━━━━━━━━━━━━━ 8s 65ms/step - accuracy: 0.9880 - loss: 0.0491 - val_accuracy: 0.9686 - val_loss: 0.0943
Epoch 3/3
126/126 ━━━━━━━━━━━━━━━━━━━━ 8s 65ms/step - accuracy: 0.9955 - loss: 0.0174 - val_accuracy: 0.9686 - val_loss: 0.0882


In [20]:
model.save("spam_lstm_model.h5")


In [23]:
import pickle

with open("tokenizer.pkl", "wb") as file:
    pickle.dump(tok, file)
